In [1]:
from langchain_community.embeddings import BaichuanTextEmbeddings
import os

# 设置API密钥
key = open('./ken_files/baichuan_API-Key.md').read().strip()
embeddings = BaichuanTextEmbeddings(api_key=key)

# 示例文本
text_1 = "今天天气不错"
text_2 = "今天阳光很好"

# 获取单个文本的嵌入
query_result = embeddings.embed_query(text_1)
print("单个文本嵌入结果:", query_result[:5])  # 只打印前5个元素

# 获取多个文本的嵌入
doc_result = embeddings.embed_documents([text_1, text_2])
print("多个文本嵌入结果:", [vec[:5] for vec in doc_result])  # 每个向量只打印前5个元素

C:\Users\wangz\miniconda3\envs\langchain\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


单个文本嵌入结果: [-0.0075800866, 0.05336674, -0.017781395, 0.035153065, 0.054738127]
多个文本嵌入结果: [[-0.0075800787, 0.053366765, -0.017781377, 0.035153035, 0.054738108], [-0.0020532398, 0.022759888, -0.04714017, 0.045531422, 0.03590641]]


In [2]:
# 这里模拟一个QA场景，我们定义一个问题，然后定义10条文本作为回答。然后分别对问题和回答各自进行词向量转换：

In [3]:
query = "早睡早起到底是不是保持身体健康的标准？"

sentences = ["早睡早起确实是保持身体健康的重要因素之一。它有助于同步我们的生物钟，并提高睡眠质量。", 
             "早睡早起可以帮助人们更好地适应自然光周期，从而优化褪黑激素的产生，这种激素是调节睡眠和觉醒的关键。",
             "关于提高工作效率，确保在日常饮食中包含充足的蛋白质、复合碳水化合物和健康脂肪非常关键。",
             "投资可再生能源项目和推广电动汽车可以显著减少温室气体排放，从而缓解气候变化带来的负面影响。",
             "多发性硬化症是一种影响中枢神经系统的自身免疫疾病，导致神经传导受损。虽然与阿尔茨海默症类似，多发性硬化症的主要症状包括疲劳、视觉障碍和肌肉控制问题。",
             "今天的天气太好了，可以早点起床去爬山",
             "如果下班特别晚的话，我建议你还是打车回家吧",
             "提升学术研究质量需侧重于多学科融合和国际合作。研究机构应该鼓励学者之间的交流，通过共享数据和研究方法，来推动科学发现和技术创新。",
             "如果你认为我说的没用，那你大可以不必理会。",
             "衡量一个人是否成功的标准在于他到底能不能让身边的人都变的优秀"

]

In [4]:
# 使用`embed_documents`方法，传入`sentences`列表，得到每条文本的向量表示

In [5]:
sentence_embeddings = embeddings.embed_documents(sentences)
# sentence_embeddings

In [6]:
# 通过`embed_query`方法生成问题的向量表示

In [7]:
embedded_query = embeddings.embed_query(query)
#embedded_query

In [8]:
# **开源EMbedding Models**
# ollama官网进行开源模型下载：https://ollama.com/search?q=embedding
# 我们以nomic-embed-text向量模型为例：

In [9]:
import requests
def ollama_embedding_by_api(text):
    res = requests.post(
        url = 'http://127.0.0.1:11434/api/embeddings',
        json = {
            "model":'nomic-embed-text:latest',
            'prompt':text
        }
    )
    embedding_list = res.json()['embedding']
    return embedding_list

In [10]:
ollama_embedding_by_api(text_1)

[0.6411043405532837,
 0.3499467670917511,
 -3.8789029121398926,
 0.06141183152794838,
 0.23303422331809998,
 1.3957492113113403,
 0.03881264477968216,
 -0.3642474114894867,
 -1.4936656951904297,
 -1.0421382188796997,
 -1.135499358177185,
 1.4623793363571167,
 -0.35445839166641235,
 -0.7944302558898926,
 0.4556824266910553,
 -0.5033971071243286,
 0.02595250867307186,
 -0.2868601083755493,
 -1.0372107028961182,
 1.827039122581482,
 -0.1278216540813446,
 0.9451901316642761,
 -0.9036160111427307,
 -1.2997734546661377,
 1.2230840921401978,
 0.26422521471977234,
 0.49909520149230957,
 1.3273905515670776,
 -1.0288593769073486,
 -0.9023132920265198,
 0.28728336095809937,
 -0.9346386790275574,
 -0.5878039002418518,
 0.547301173210144,
 -1.6004823446273804,
 -0.008258329704403877,
 0.2697693109512329,
 0.8276067972183228,
 0.5450313687324524,
 0.28160396218299866,
 1.0255005359649658,
 -0.32273802161216736,
 -0.5317718982696533,
 -0.09272128343582153,
 0.7616119384765625,
 -0.4557165205478668,
 

In [11]:
# **代码构建简易RAG**

In [12]:
import uuid
import chromadb
import requests
import os
from openai import OpenAI

#创建数据库，类似创建一个文件夹
client = chromadb.PersistentClient(path="./db/chroma_demo")
#创建数据集合（库表）
collection = client.get_or_create_collection(name="collection_v2")

In [13]:
#数据集切分-分块处理
def file_chunk_list():
    #1.读取文件内容
    with open('data/中医问诊.txt','r',encoding='utf-8') as fp:
        data = fp.read()
    #2.根据换行切割:将一个病症作为一个列表元素数据
    chunk_list = data.split('\n\n')
    chunk_list = [chunk for chunk in chunk_list if chunk]
    return chunk_list

In [14]:
file_chunk_list()

['风寒感冒\n症状：恶寒重，发热轻，无汗，头痛，肢节酸痛，鼻塞声重，或鼻痒喷嚏，时流清涕，咽痒，咳嗽，咳痰稀薄色白，口不渴或渴喜热饮，舌苔薄白，脉浮紧。\n药方：荆防败毒散。药物组成包括荆芥、防风、羌活、独活、柴胡、前胡、川芎、枳壳、茯苓、桔梗、甘草等，具有辛温解表的功效。',
 '风热感冒\n症状：发热，微恶风，有汗，头胀痛，鼻塞流黄涕，咳嗽，痰黏或黄，咽燥红肿，口渴，舌尖边红，苔薄黄，脉浮数。\n药方：银翘散。药物组成有金银花、连翘、桔梗、薄荷、竹叶、生甘草、荆芥穗、淡豆豉、牛蒡子等，能辛凉解表、清热解毒。',
 '暑湿感冒\n症状：发热，微恶风，汗少，汗出热不退，肢体酸重或疼痛，头昏重胀痛，鼻流浊涕，心烦口渴，胸闷脘痞，泛恶欲呕，小便短赤，舌苔薄黄腻，脉濡数。\n药方：新加香薷饮。药物包括香薷、银花、鲜扁豆花、厚朴、连翘等，可解表散邪、清热化湿。',
 '风寒咳嗽\n症状：咳嗽声重，气急，咽痒，咳痰稀薄色白，常伴鼻塞，流清涕，头痛，肢体酸楚，恶寒发热，无汗，舌苔薄白，脉浮 或浮紧。\n药方：止嗽散。药物组成有桔梗、荆芥、紫菀、百部、白前、甘草、陈皮等，用于宣利肺气、疏风止咳。',
 '风热咳嗽\n症状：咳嗽频剧，气粗或咳声嘶哑，喉燥咽痛，咳痰不爽，痰黏稠或黄，咳时汗出，常伴鼻流黄涕，口渴，头痛，肢楚，身热恶风，舌苔薄黄，脉浮数或浮滑。\n药方：桑菊饮。药物包含桑叶、菊花、杏仁、连翘、薄荷、苦桔梗、芦根等，能疏风清热、宣肺止咳。',
 '痰湿咳嗽\n症状：咳嗽反复发作，咳声重浊，痰多，因痰而嗽，痰黏腻或稠厚成块，色白或带灰色，胸闷脘痞，呕恶食少，体倦，大便时溏，舌苔白腻，脉濡滑。\n药方：二陈平胃散。药物由陈皮、半夏、茯苓、苍术、甘草、厚朴等组成，起燥湿化痰、理气和中之功。',
 '阴虚咳嗽\n症状：干咳，咳声短促，痰少黏白，或痰中带血丝，口咽干燥，或声音逐渐嘶哑，午后潮热，手足心热，夜寐盗汗，两颧发红，舌红少苔，脉细数。\n药方：沙参麦冬汤。药物包括沙参、麦冬、玉竹、天花粉、百合、甘草等，有滋阴润肺之效。',
 '哮喘（冷哮）\n症状：喉中哮鸣有声，胸膈满闷，咳痰稀白，面色晦滞带青，口不渴，或渴喜热饮，天冷或受寒加重，形寒肢冷，舌苔白滑，脉弦滑或浮紧。\n药方：射干麻黄汤。药物组成为射干、麻黄、生姜、细辛、紫菀、款冬花、大枣、五味子、半夏等，可温肺散寒、化痰

In [15]:
#数据集向量化封装
def ollama_embedding_by_api(text):
    #使用nomic向量模型
    # res = requests.post(
    #     url = 'http://127.0.0.1:11434/api/embeddings',
    #     json = {
    #         "model":'nomic-embed-text:latest',
    #         'prompt':text
    #     }
    # )
    # embedding_list = res.json()['embedding']
    # return embedding_list
    
    #使用阿里百炼向量模型（效果超级好）
    API_KEY = open('./ken_files/bailiann_API-Key.md',encoding='utf-8').read().strip()
    client = OpenAI(
        api_key=API_KEY,  # 如果您没有配置环境变量，请在此处用您的API Key进行替换
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"  # 百炼服务的base_url
    )

    completion = client.embeddings.create(
        model="text-embedding-v3",
        input=text,
        dimensions=1024,
        encoding_format="float"
    )
    return completion.data[0].embedding

In [16]:
#deepseek模型调用
def ollama_generate_by_api(prompt):
    res = requests.post(
    url = 'http://127.0.0.1:11434/api/generate',
    json = {
            "model":'deepseek-r1:7b',
            'prompt':prompt,
            'stream':False
        }
    )
    res = res.json()['response']
    return res


In [17]:
#整体集成
def initial():
    #构造数据
    documents = file_chunk_list()
    #给每一个数据创建唯一的id标识
    ids = [str(uuid.uuid4()) for _ in documents]
    embeddings = [ollama_embedding_by_api(text) for text in documents]

    #插入数据
    collection.add(
        ids = ids,
        documents=documents,
        embeddings=embeddings
    )

In [18]:
def run():
    qs = '我好像是感冒了，症状是头痛、轻微发烧、肢节酸痛、打喷嚏和流鼻涕。'
    qs_embedding = ollama_embedding_by_api(qs)
    #n_results表示匹配几个最高相似度的结果
    res = collection.query(query_embeddings=[qs_embedding,],query_texts=qs,n_results=2)
    result = res['documents'][0]
    context = '\n'.join(result)
    prompt = f'''你是一个中医问答机器人，任务是根据参考信息回答用户问题，如果你参考信息不足以回答用户问题，请回复不知道，切记不要去杜撰和自由发挥任何内容和信息，请用中文回答，参考信息：{context},来回答问题:{qs},'''
    result = ollama_generate_by_api(prompt)
    print(result)

In [19]:
initial() #执行一次即可

In [20]:
run() #可多次测试

<think>
嗯，我现在需要处理一个用户的问题，他描述了一些症状，看起来像是风寒感冒。首先，我要确认用户提供的信息是否足够详细，以确定正确的药方。

用户的症状包括：头痛、轻微发热、肢节酸痛、打喷嚏和流鼻涕。这些症状符合风寒感冒的典型表现，比如恶寒重、发热轻、无汗、鼻塞等。参考信息中提到的荆防败毒散正好适用于这种症状，因为它具有辛温解表的功效。

接下来，我需要判断用户是否还有其他症状，比如咳嗽、咳痰稀薄、口渴或舌苔的情况。如果这些症状没有提及，那么可能不需要考虑其他药物，或者可以建议用户注意观察是否有其他变化。

此外，我应该询问用户提供更多信息，如天气情况、饮食习惯和用药史，以便提供更准确的诊断和建议。这样可以帮助确认是否是风寒感冒，而不是湿热或别的感冒类型。

最后，我要确保回复符合要求，只基于提供的参考信息，不杜撰任何内容，并用中文回答。
</think>

根据您描述的症状，包括头痛、轻微发热、肢节酸痛、打喷嚏和流鼻涕，这些症状符合风寒感冒的表现。建议参考的药方是荆防败毒散，它适用于类似症状，具有辛温解表的功效。如果您还有其他症状或需要进一步的帮助，请提供更多信息以便更好地指导您。
